# Example 8 - Mixed Type Data with Missingness || Dummy Dataset

In [2]:
import os, random, warnings
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")


import numpy as np
import pandas as pd
import torch
from bstabdiff import BSTabDiff, FeatureSpec


SEED = 42
N = 80
N_CONT = 200
N_CAT = 10
N_SYN = 100
DEVICE = "cuda:0" if torch.cuda.is_available() else "cpu"


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def select_default_M(n, m):
    rho = m / n
    kappa = n * m
    if m <= 64:
        return m
    if rho < 10:
        return 64
    if kappa < 2.5e5:
        M = 32
    elif kappa < 1e6:
        M = 64
    elif kappa < 3e6:
        M = 128
    else:
        M = 192
    return min(M, m)


def make_dummy_mixed_data():
    rng = np.random.default_rng(SEED)
    y = rng.integers(0, 2, size=N)
    z = rng.normal(size=N)


    data = {}
    for j in range(N_CONT):
        scale = 0.3 + 0.7 * rng.random()
        signal = (0.2 + 0.8 * rng.random()) * y
        data[f"cont_{j}"] = scale * z + signal + rng.normal(0, 0.7, N)


    cat_levels = {
        "sex": ["Female", "Male"],
        "smoking": ["Never", "Former", "Current"],
        "stage": ["I", "II", "III", "IV"],
        "site": ["A", "B", "C"],
        "therapy": ["None", "DrugA", "DrugB"],
        "status": ["Negative", "Positive"],
        "group": ["G1", "G2", "G3", "G4"],
        "region": ["North", "South", "East", "West"],
        "marker": ["Low", "Medium", "High"],
        "batch": ["B1", "B2", "B3"]
    }


    for col, levels in cat_levels.items():
        K = len(levels)
        codes = (rng.integers(0, K, size=N) + y) % K
        data[col] = np.asarray(levels, dtype=object)[codes]


    df = pd.DataFrame(data)
    df["label"] = y


    cont_cols = [f"cont_{j}" for j in range(N_CONT)]
    cat_cols = list(cat_levels)


    for col in cont_cols:
        p = 0.08 + 0.07 * y
        miss = rng.random(N) < p
        df.loc[miss, col] = np.nan


    for col in cat_cols:
        p = 0.10 + 0.10 * y
        miss = rng.random(N) < p
        df.loc[miss, col] = np.nan


    return df, cont_cols, cat_cols


def encode_mixed_data(df, cont_cols, cat_cols):
    feature_names = cont_cols + cat_cols
    X = np.full((len(df), len(feature_names)), np.nan, dtype=np.float32)
    feature_specs = []
    category_maps = {}


    for j, col in enumerate(feature_names):
        if col in cat_cols:
            codes, levels = pd.factorize(df[col], sort=True)
            obs = codes >= 0
            X[obs, j] = codes[obs].astype(np.float32)
            category_maps[col] = list(levels)
            feature_specs.append(
                FeatureSpec(name=col, kind="categorical", n_categories=len(levels))
            )
        else:
            X[:, j] = pd.to_numeric(df[col], errors="coerce").to_numpy(dtype=np.float32)
            feature_specs.append(FeatureSpec(name=col, kind="continuous"))


    y = df["label"].to_numpy(dtype=int)
    return X, y, feature_names, feature_specs, category_maps


def decode_synthetic(X_syn, y_syn, feature_names, category_maps):
    df_syn = pd.DataFrame(X_syn, columns=feature_names)
    for col, levels in category_maps.items():
        vals = df_syn[col].to_numpy(dtype=float)
        obs = np.isfinite(vals)
        decoded = np.full(len(vals), np.nan, dtype=object)
        codes = np.rint(vals[obs]).astype(int)
        decoded[obs] = [levels[c] for c in codes]
        df_syn[col] = decoded
    if y_syn is not None:
        df_syn["label"] = np.asarray(y_syn).astype(int)
    return df_syn


set_seed(SEED)


df_real, continuous_cols, categorical_cols = make_dummy_mixed_data()
X, y, feature_names, feature_specs, category_maps = encode_mixed_data(
    df_real, continuous_cols, categorical_cols
)


n, m = X.shape
M = select_default_M(n, m)


print(f"Dataset: n={n}, m={m}, rho={m/n:.2f}, kappa={n*m}, M={M}")
print(f"Continuous features: {len(continuous_cols)}")
print(f"Categorical features: {len(categorical_cols)}")
print(f"Classes: {np.unique(y).tolist()}")
print(f"Missing cells: {np.isnan(X).sum()} ({100*np.isnan(X).mean():.2f}%)")
print(f"Device: {DEVICE}")


print("\nCategorical mappings:")
for col, levels in category_maps.items():
    print(f"{col}: {levels}")


model = BSTabDiff(
    feature_specs=feature_specs,
    M=M,
    blocks=None,
    permute_features=False,
    prior_type="diffusion",
    device=DEVICE,
    random_state=SEED,
    prior_epochs=300,
    prior_batch=64,
    prior_lr=1e-3,
    verbose_every=100,
    save_dir=None,
    save_name="bstabdiff_dummy_mixed",
    save_best=True,
    use_ema=True,
    ema_decay=0.999,
    use_gobs=False,
    use_gobs_fc=False
)


model.fit(X, y)
X_syn, R_syn, y_syn = model.sample(n_samples=N_SYN, return_mask=True)


X_syn = np.asarray(X_syn, dtype=np.float32)
R_syn = np.asarray(R_syn)
y_syn = np.asarray(y_syn).astype(int)


print("\nSynthetic output:")
print("X_syn:", X_syn.shape)
print("R_syn:", R_syn.shape)
print("y_syn:", y_syn.shape)
print(f"Synthetic missing cells: {np.isnan(X_syn).sum()} ({100*np.isnan(X_syn).mean():.2f}%)")


for j, fs in enumerate(feature_specs):
    if fs.kind == "categorical":
        vals = X_syn[:, j]
        vals = vals[np.isfinite(vals)]
        assert np.all(vals == np.floor(vals))
        assert np.all((vals >= 0) & (vals < fs.n_categories))


df_syn = decode_synthetic(X_syn, y_syn, feature_names, category_maps)


print("\nReal mixed data:")
print(df_real[continuous_cols[:3] + categorical_cols[:4] + ["label"]].head())


print("\nSynthetic mixed data:")
print(df_syn[continuous_cols[:3] + categorical_cols[:4] + ["label"]].head())


df_real.to_csv("bstabdiff_dummy_mixed_real.csv", index=False)
df_syn.to_csv("bstabdiff_dummy_mixed_synthetic.csv", index=False)
pd.DataFrame(R_syn, columns=feature_names).to_csv(
    "bstabdiff_dummy_mixed_mask.csv", index=False
)


print("\nSaved:")
print("bstabdiff_dummy_mixed_real.csv")
print("bstabdiff_dummy_mixed_synthetic.csv")
print("bstabdiff_dummy_mixed_mask.csv")

Dataset: n=80, m=210, rho=2.62, kappa=16800, M=64
Continuous features: 200
Categorical features: 10
Classes: [0, 1]
Missing cells: 1977 (11.77%)
Device: cuda:0

Categorical mappings:
sex: ['Female', 'Male']
smoking: ['Current', 'Former', 'Never']
stage: ['I', 'II', 'III', 'IV']
site: ['A', 'B', 'C']
therapy: ['DrugA', 'DrugB', 'None']
status: ['Negative', 'Positive']
group: ['G1', 'G2', 'G3', 'G4']
region: ['East', 'North', 'South', 'West']
marker: ['High', 'Low', 'Medium']
batch: ['B1', 'B2', 'B3']
[prior:diffusion] epoch 1/300 | loss=0.991440
[prior:diffusion] epoch 100/300 | loss=0.580394
[prior:diffusion] epoch 200/300 | loss=0.456283
[prior:diffusion] epoch 300/300 | loss=0.470165

Synthetic output:
X_syn: (100, 210)
R_syn: (100, 210)
y_syn: (100,)
Synthetic missing cells: 2554 (12.16%)

Real mixed data:
     cont_0    cont_1    cont_2     sex smoking stage site  label
0  0.420908       NaN  0.917227  Female   Never     I    A      0
1  1.679130  0.921090  1.413734    Male  Former